# Travel Query Intent Classification Using NLP

## End-to-end notebook

**Team:** Krishnav Rastogi (Lead), Vinayak Goyal, Sakaar Vir Singh

This notebook takes one or more labeled travel-query CSV files, checks the data, prepares a leakage-safe train/test setup, compares several text-classification models, selects a final model using cross-validation on the training data, evaluates that model once on the held-out test set, and exports a self-contained bundle for the Streamlit dashboard.

The notebook is designed for Google Colab. CSV files are uploaded here in Colab and are never uploaded directly to the Streamlit dashboard.

### Run order

Run the cells from top to bottom. The only configuration you normally need to change is the **Configuration** cell.

### What the notebook produces

- dataset and data-quality reports
- processed train/test CSVs
- model comparison results
- final-model metrics
- confusion matrix and per-intent performance
- error analysis sample
- subgroup audit
- feature-level explanation for TF-IDF models
- a small prediction demo
- a focused set of presentation-ready graphs (about ten, covering data, models, errors, and uncertainty)
- `travel_nlp_run_bundle.zip` for the Streamlit dashboard

## 1. Environment setup

The first cells install the small set of libraries used by the core pipeline and create a clean working directory.

The core models do not require a large language model or an external API. An optional Sentence-BERT benchmark can be enabled later in the configuration cell.

In [ ]:
# Core dependencies
!pip -q install "pandas>=2,<3" "numpy>=1.24,<3" "matplotlib>=3.7,<4" "scikit-learn>=1.3,<2"

print("Core dependencies installed.")

In [ ]:
from pathlib import Path
import os
import shutil

for folder in ["data/processed", "results", "model_card"]:
    shutil.rmtree(folder, ignore_errors=True)

for folder in ["data/raw", "data/processed", "results/figures", "model_card"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Working folders are ready.")

## 2. Upload the dataset

Upload all CSV files that belong to this experiment **here in Google Colab**. The dashboard does not accept raw CSV files; it accepts only the generated `travel_nlp_run_bundle.zip`.

The pipeline accepts:
- an existing train/test pair such as `train_atis.csv` and `test_atis.csv`
- one unsplit labeled dataset
- several datasets with different column names
- a mixture of pre-split and unsplit files

Column names do not have to match ATIS. The next configuration cell tries to identify the text and label columns automatically and stops when the choice is genuinely ambiguous.

If there are **ambiguous text columns** or **ambiguous label columns**, the notebook stops rather than silently choosing one; use `COLUMN_OVERRIDES` in the configuration cell when needed.


In [ ]:
import shutil

try:
    from google.colab import files

    # Start each Colab run with a clean raw-data staging area.
    for old_file in Path("data/raw").glob("*"):
        if old_file.is_file():
            old_file.unlink()

    uploaded = files.upload()

    for filename in uploaded:
        if filename.lower().endswith(".csv"):
            shutil.move(filename, Path("data/raw") / filename)
        else:
            print(f"Skipped {filename!r}: only CSV files are accepted.")

    print("CSV upload is handled in Colab. The Streamlit dashboard receives only the completed travel_nlp_run_bundle.zip.")
    print("Files staged in data/raw:")
    for path in sorted(Path("data/raw").glob("*.csv")):
        print(" -", path.name)

except ImportError:
    print("Colab upload controls are unavailable in this environment.")
    print("Place your CSV files in data/raw/ and continue.")

## 3. Configuration and automatic column detection

Most datasets can be used without changing the code.

The detector looks for common text and label names first, then applies conservative type/content checks. If two columns are similarly plausible, the notebook stops instead of making a silent choice.

Use `COLUMN_OVERRIDES` when a file needs an explicit mapping.

The model search below is intentionally lightweight and reproducible: several classical NLP pipelines are compared with stratified cross-validation, and the final model is selected using **training-only CV Macro-F1**. The held-out test set is used only for final evaluation.

In [ ]:
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path

TEXT_NAMES = [
    "text", "query", "user_query", "utterance", "sentence",
    "question", "input", "prompt", "message", "request", "text_data"
]
LABEL_NAMES = [
    "intent", "label", "intent_label", "category", "class",
    "target", "y", "topic", "intent_name"
]

# Optional manual mappings. Example:
# COLUMN_OVERRIDES = {
#     "my_dataset.csv": {"text_col": "query", "label_col": "category", "split": None}
# }
COLUMN_OVERRIDES = {}

# General experiment settings
LABEL_HARMONIZATION_MAP = {}
DEDUPLICATE_EXACT_TEXT = True
FAIL_ON_PRE_SPLIT_TEXT_OVERLAP = True

RANDOM_SEED = 42
CV_FOLDS = 5

# Optional semantic benchmark. It is not used for final model selection.
RUN_SBERT = False
SBERT_MODEL_NAME = "all-MiniLM-L6-v2"

def _normalised_columns(df):
    return {str(column).strip().lower(): column for column in df.columns}

def _detect_columns(path):
    sample = pd.read_csv(path, nrows=5000)
    columns = _normalised_columns(sample)

    text_hits = [
        (columns[name], 0.99 - index * 0.01)
        for index, name in enumerate(TEXT_NAMES)
        if name in columns
    ]
    label_hits = [
        (columns[name], 0.99 - index * 0.01)
        for index, name in enumerate(LABEL_NAMES)
        if name in columns
    ]

    # Conservative fallback for an unrecognised text column.
    if not text_hits:
        for column in sample.columns:
            if str(sample[column].dtype) not in {"object", "string"}:
                continue
            values = sample[column].dropna().astype(str)
            if len(values) < 2:
                continue
            average_length = float(values.str.len().mean())
            whitespace_rate = float(values.str.contains(r"\s", regex=True).mean())
            if average_length >= 8 and whitespace_rate >= 0.20:
                score = min(0.88, 0.35 + min(average_length / 80, 0.35) + 0.18 * whitespace_rate)
                text_hits.append((column, score))

    # Conservative fallback for an intent/category column.
    if not label_hits:
        for column in sample.columns:
            if text_hits and column == text_hits[0][0]:
                continue
            if str(sample[column].dtype) not in {"object", "string", "category"}:
                continue
            unique_count = int(sample[column].nunique(dropna=True))
            if 2 <= unique_count <= min(1000, max(2, int(len(sample) * 0.25))):
                label_hits.append((column, 0.55))

    text_hits = sorted(set(text_hits), key=lambda item: (-item[1], str(item[0])))
    label_hits = sorted(set(label_hits), key=lambda item: (-item[1], str(item[0])))

    if not text_hits or not label_hits:
        raise ValueError(
            f"{path}: could not identify both a text and label column. "
            f"Columns found: {sample.columns.tolist()}. Add an entry to COLUMN_OVERRIDES."
        )

    if len(text_hits) > 1 and text_hits[1][1] >= text_hits[0][1] - 0.08:
        raise ValueError(f"{path}: text-column detection is ambiguous: {text_hits[:4]}")

    if len(label_hits) > 1 and label_hits[1][1] >= label_hits[0][1] - 0.08:
        raise ValueError(f"{path}: label-column detection is ambiguous: {label_hits[:4]}")

    filename = Path(path).stem.lower()
    if re.search(r"(^|[_\-.])train([_\-.]|$)", filename):
        split = "train"
    elif re.search(r"(^|[_\-.])test([_\-.]|$)", filename):
        split = "test"
    else:
        split = None

    return text_hits[0][0], label_hits[0][0], split

DATASETS = []
raw_paths = sorted(Path("data/raw").glob("*.csv"))

if not raw_paths:
    raise ValueError("No CSV files found in data/raw/. Upload at least one labeled CSV.")

for path in raw_paths:
    if path.name in COLUMN_OVERRIDES:
        config = {"path": str(path), **COLUMN_OVERRIDES[path.name]}
    else:
        text_col, label_col, split = _detect_columns(str(path))
        config = {
            "path": str(path),
            "text_col": text_col,
            "label_col": label_col,
            "split": split,
            "split_inferred_from_filename": split is not None,
        }
    DATASETS.append(config)

# A split guessed from a filename is only trusted when BOTH a train and a test file are present.
# Otherwise names such as "train_rail_queries.csv" (train = railway) would be mistaken for a
# training split with no test set. An explicit split in COLUMN_OVERRIDES is always respected.
_inferred = {cfg["split"] for cfg in DATASETS if cfg.get("split_inferred_from_filename")}
if _inferred and _inferred != {"train", "test"}:
    for cfg in DATASETS:
        if cfg.get("split_inferred_from_filename"):
            print(
                f"NOTE: {Path(cfg['path']).name} looks like a {cfg['split']!r} file by name, but no matching "
                "train/test counterpart was uploaded. Treating it as an unsplit dataset and splitting it "
                "in the notebook. Set the split explicitly in COLUMN_OVERRIDES to override."
            )
            cfg["split"] = None

print("Dataset configuration")
print("-" * 72)
for config in DATASETS:
    print(
        f"{Path(config['path']).name}: "
        f"text={config['text_col']!r}, label={config['label_col']!r}, split={config['split']!r}"
    )

## 4. Load, clean, and combine the datasets

Every input file is normalised to two modelling columns, `text` and `intent`, while the original filename is retained as `source_dataset`.

Before a model is fitted, the notebook checks for:
- missing or empty text/labels
- conflicting labels for the same normalised query
- duplicate queries
- train/test text overlap
- rare classes
- source-level differences

A conflicting label is treated as a data-quality issue and stops the run. That is safer than silently choosing one of the labels.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

TEXT_COL, INTENT_COL = "text", "intent"

def load_one(config):
    frame = pd.read_csv(config["path"])

    if config["text_col"] not in frame.columns or config["label_col"] not in frame.columns:
        raise ValueError(
            f"{config['path']}: expected columns "
            f"{config['text_col']!r} and {config['label_col']!r}. "
            f"Available columns: {frame.columns.tolist()}"
        )

    keep_columns = [config["text_col"], config["label_col"]]
    stable_candidates = ["row_id", "id", "query_id", "sample_id", "record_id"]
    stable_column = next((c for c in stable_candidates if c in frame.columns and c not in keep_columns), None)
    if stable_column:
        keep_columns.append(stable_column)

    output = frame[keep_columns].rename(
        columns={config["text_col"]: TEXT_COL, config["label_col"]: INTENT_COL}
    )
    output["source_dataset"] = Path(config["path"]).name
    # Keep a deterministic row identifier even when the source CSV has no ID column.
    if stable_column is None:
        output["record_id"] = [f"{Path(config['path']).name}::row_{i}" for i in range(len(output))]
    elif stable_column != "record_id":
        output["record_id"] = output[stable_column].astype(str).map(lambda value: f"{Path(config['path']).name}::{value}")
    output["_split"] = config["split"]
    return output

frames = [load_one(config) for config in DATASETS]
combined = pd.concat(frames, ignore_index=True)

before_clean = len(combined)
combined = combined.dropna(subset=[TEXT_COL, INTENT_COL]).copy()
combined[TEXT_COL] = combined[TEXT_COL].astype(str).str.strip()
combined[INTENT_COL] = combined[INTENT_COL].astype(str).str.strip()
combined = combined[(combined[TEXT_COL] != "") & (combined[INTENT_COL] != "")]
print(f"Removed {before_clean - len(combined)} rows with missing or empty text/labels.")

if LABEL_HARMONIZATION_MAP:
    combined[INTENT_COL] = combined[INTENT_COL].replace(LABEL_HARMONIZATION_MAP)

def text_key(series):
    return (
        series.astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

combined["_text_key"] = text_key(combined[TEXT_COL])

conflict_counts = combined.groupby("_text_key")[INTENT_COL].nunique()
conflicts = conflict_counts[conflict_counts > 1]

if len(conflicts):
    examples = conflicts.index.tolist()[:5]
    raise ValueError(
        f"Found {len(conflicts)} normalised queries with conflicting labels. "
        f"Examples: {examples}"
    )

duplicate_count_before = int(combined.duplicated(["_text_key", INTENT_COL]).sum())
cross_source_count = int((combined.groupby("_text_key")["source_dataset"].nunique() > 1).sum())

if DEDUPLICATE_EXACT_TEXT:
    combined = combined.drop_duplicates(
        subset=["_text_key", INTENT_COL], keep="first"
    ).reset_index(drop=True)

print(f"Duplicate text/label rows found before deduplication: {duplicate_count_before}")
print(f"Normalised queries appearing in multiple source files: {cross_source_count}")

pre_split = combined[combined["_split"].isin(["train", "test"])].copy()
unsplit = combined[combined["_split"].isna()].copy()

train = (
    pre_split[pre_split["_split"] == "train"]
    .drop(columns=["_split"])
    .reset_index(drop=True)
)
test = (
    pre_split[pre_split["_split"] == "test"]
    .drop(columns=["_split"])
    .reset_index(drop=True)
)

train_keys = set(text_key(train[TEXT_COL]))
test_keys = set(text_key(test[TEXT_COL]))
pre_split_overlap = train_keys & test_keys

if pre_split_overlap and FAIL_ON_PRE_SPLIT_TEXT_OVERLAP:
    raise ValueError(
        f"Train/test leakage detected: {len(pre_split_overlap)} normalised queries "
        "occur in both supplied splits."
    )

if len(unsplit):
    occupied = train_keys | test_keys
    unsplit = unsplit[~unsplit["_text_key"].isin(occupied)].copy()

    counts = unsplit[INTENT_COL].value_counts()
    singleton_labels = counts[counts < 2].index
    singleton_rows = unsplit[unsplit[INTENT_COL].isin(singleton_labels)]
    splittable = unsplit[~unsplit[INTENT_COL].isin(singleton_labels)]

    if len(splittable):
        test_fraction = min(0.5, max(0.2, splittable[INTENT_COL].nunique() / len(splittable)))
        split_train, split_test = train_test_split(
            splittable,
            test_size=test_fraction,
            stratify=splittable[INTENT_COL],
            random_state=RANDOM_SEED,
        )
        train = pd.concat([train, split_train], ignore_index=True)
        test = pd.concat([test, split_test], ignore_index=True)

    if len(singleton_rows):
        train = pd.concat([train, singleton_rows], ignore_index=True)
        print(f"Kept {len(singleton_rows)} singleton-class rows in training only.")

for frame in (train, test):
    frame.drop(columns=["_text_key"], inplace=True, errors="ignore")

if train.empty or test.empty:
    raise ValueError(
        f"Training/test split is not usable: train={len(train)}, test={len(test)}."
    )

print(f"Final training rows: {len(train):,}")
print(f"Final test rows:     {len(test):,}")
print(f"Training intents:    {train[INTENT_COL].nunique():,}")
print(f"Test intents:        {test[INTENT_COL].nunique():,}")

## 5. Data profile and exploratory checks

The next cells create the basic dataset evidence used by both the notebook report and the Streamlit dashboard.

The plots are deliberately simple: class distribution, source distribution, and query length are the first things to inspect before interpreting model scores.

In [ ]:
import matplotlib.pyplot as plt

# Figure palette: the same colours the Studio uses, so notebook figures and Studio pages look like one product.
# Presentation only. No number, file name or table below depends on these colours.
STUDIO_COLORS = {"accent": "#165FC5", "right": "#07665C", "wrong": "#C0332D", "warn": "#8F5B00", "muted": "#6B7785", "ink": "#16202A", "grid": "#D9E0E7"}
from matplotlib.colors import LinearSegmentedColormap
STUDIO_CMAP = LinearSegmentedColormap.from_list("studio_right", ["#FFFFFF", "#8FD1C6", STUDIO_COLORS["right"]])
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": STUDIO_COLORS["muted"],
                     "axes.labelcolor": STUDIO_COLORS["ink"], "xtick.color": STUDIO_COLORS["ink"], "ytick.color": STUDIO_COLORS["ink"],
                     "axes.titleweight": "bold", "axes.grid": True, "grid.color": STUDIO_COLORS["grid"], "grid.linewidth": 0.8,
                     "axes.axisbelow": True, "figure.facecolor": "white", "axes.facecolor": "white"})

train["word_count"] = train[TEXT_COL].str.split().str.len()
test["word_count"] = test[TEXT_COL].str.split().str.len()

profile = {
    "training_samples": int(len(train)),
    "test_samples": int(len(test)),
    "training_intents": int(train[INTENT_COL].nunique()),
    "test_intents": int(test[INTENT_COL].nunique()),
    "source_datasets": train["source_dataset"].value_counts().to_dict(),
    "duplicate_texts_train": int(train[TEXT_COL].duplicated().sum()),
    "duplicate_rows_removed": duplicate_count_before,
    "cross_source_duplicate_queries": cross_source_count,
    "average_query_length": round(float(train["word_count"].mean()), 2),
    "median_query_length": float(train["word_count"].median()),
    "train_test_text_overlap": int(len(pre_split_overlap)),
    "conflicting_duplicate_labels": int(len(conflicts)),
}

Path("results/dataset_profile.json").write_text(
    json.dumps(profile, indent=2), encoding="utf-8"
)

print("Training data summary")
print("-" * 40)
print(pd.Series(profile))

In [ ]:
counts = train[INTENT_COL].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

counts.sort_values().plot(kind="barh", ax=axes[0], color=STUDIO_COLORS["accent"])
axes[0].set_title("Training intent distribution")
axes[0].set_xlabel("Queries")
axes[0].set_ylabel("Intent")

train["source_dataset"].value_counts().sort_values().plot(kind="barh", ax=axes[1], color=STUDIO_COLORS["muted"])
axes[1].set_title("Training rows by source dataset")
axes[1].set_xlabel("Rows")
axes[1].set_ylabel("Source")

plt.tight_layout()
plt.savefig("results/figures/intent_and_dataset_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Smallest training classes:")
print(counts.sort_values().head(10))

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(train["word_count"], bins=30, edgecolor="white", color=STUDIO_COLORS["accent"])
median_length = train["word_count"].median()
plt.axvline(median_length, linestyle="--", color=STUDIO_COLORS["warn"], label=f"Median = {median_length:.0f} words")
plt.title("Training query length")
plt.xlabel("Words per query")
plt.ylabel("Number of queries")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/query_length_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

### Split checks

A model cannot learn an intent that is completely absent from its training data. The next check records any test-only intents so that a low score caused by the split is not mistaken for a modelling failure.

In [ ]:
test_only_intents = sorted(set(test[INTENT_COL]) - set(train[INTENT_COL]))
rare_training_classes = train[INTENT_COL].value_counts()
rare_training_classes = rare_training_classes[rare_training_classes < 2]

print("Test-only intents:", test_only_intents or "None")
print("Training intents with fewer than 2 examples:")
print(rare_training_classes if len(rare_training_classes) else "None")

train_texts = set(text_key(train[TEXT_COL]))
test_texts = set(text_key(test[TEXT_COL]))
text_overlap = train_texts & test_texts

print(f"Normalised train/test text overlap: {len(text_overlap)}")

### Pre-training data-quality gate

Everything below is checked **before any model is fitted**. Hard problems (empty splits, a test intent that training never saw is only a warning) stop the run; softer findings are written to `results/data_quality.json` so the Studio can show them next to the metrics.

In [ ]:
# PRE-TRAINING DATA QUALITY GATE (runs before any model is fitted)
_tk = lambda s: s.astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
_train_counts = train[INTENT_COL].value_counts()
_dup_train = int(_tk(train[TEXT_COL]).duplicated().sum())
_dup_test = int(_tk(test[TEXT_COL]).duplicated().sum())
_overlap = len(set(_tk(train[TEXT_COL])) & set(_tk(test[TEXT_COL])))
data_quality = {
    "train_rows": int(len(train)), "test_rows": int(len(test)),
    "train_intents": int(train[INTENT_COL].nunique()), "test_intents": int(test[INTENT_COL].nunique()),
    "duplicate_train_queries": _dup_train, "duplicate_test_queries": _dup_test,
    "train_test_exact_overlap": int(_overlap),
    "conflicting_label_queries": int(len(conflicts)),
    "largest_class": {"intent": str(_train_counts.index[0]), "rows": int(_train_counts.iloc[0])},
    "smallest_class": {"intent": str(_train_counts.index[-1]), "rows": int(_train_counts.iloc[-1])},
    "imbalance_ratio": round(float(_train_counts.iloc[0] / max(_train_counts.iloc[-1], 1)), 2),
    "classes_under_5_train_rows": [str(k) for k, v in _train_counts.items() if v < 5],
    "test_only_intents": sorted(map(str, set(test[INTENT_COL]) - set(train[INTENT_COL]))),
    "sources": sorted(map(str, train["source_dataset"].unique())),
}
_gate = [
    ("Training set not empty", len(train) > 0, True),
    ("Test set not empty", len(test) > 0, True),
    ("No conflicting labels", len(conflicts) == 0, True),
    ("No train/test text overlap", _overlap == 0, True),
    ("No test-only intents", not data_quality["test_only_intents"], False),
    ("No duplicate test queries", _dup_test == 0, False),
]
print("PRE-TRAINING DATA QUALITY GATE")
print("-" * 60)
_hard_fail = []
for _name, _ok, _hard in _gate:
    print(f"{'PASS' if _ok else ('FAIL' if _hard else 'WARN'):<6} {_name}")
    if not _ok and _hard: _hard_fail.append(_name)
print(f"Class imbalance (largest / smallest): {data_quality['imbalance_ratio']}x; classes with <5 training rows: {len(data_quality['classes_under_5_train_rows'])}")
# Near-duplicate leakage: test queries almost identical to a training query (cosine >= 0.9 on word 1-2gram TF-IDF).
# Exact overlap is already a hard failure above; near-duplicates are reported, not blocked.
from sklearn.feature_extraction.text import TfidfVectorizer as _TV
_nd_vec = _TV(ngram_range=(1, 2), sublinear_tf=True).fit(pd.concat([train[TEXT_COL], test[TEXT_COL]]).astype(str).str.lower())
_sim = (_nd_vec.transform(test[TEXT_COL].astype(str).str.lower()) @ _nd_vec.transform(train[TEXT_COL].astype(str).str.lower()).T).tocsr()
_best_i = np.asarray(_sim.argmax(axis=1)).ravel()
_best_s = np.asarray(_sim.max(axis=1).todense()).ravel()
_near = np.where(_best_s >= 0.9)[0]
_same = [bool(test[INTENT_COL].iloc[i] == train[INTENT_COL].iloc[_best_i[i]]) for i in _near]
data_quality["near_duplicates"] = {
    "method": "cosine similarity of word 1-2gram TF-IDF vectors", "threshold": 0.9,
    "test_rows_with_near_train_match": int(len(_near)),
    "share_of_test": round(float(len(_near) / max(len(test), 1)), 4),
    "pairs_with_different_labels": int(sum(not s for s in _same)),
    "test_queries": [str(test[TEXT_COL].iloc[i]) for i in _near[:2000]],
    "examples": [{"test": str(test[TEXT_COL].iloc[i]), "train": str(train[TEXT_COL].iloc[_best_i[i]]),
                  "similarity": round(float(_best_s[i]), 3), "same_label": bool(ok)}
                 for i, ok in sorted(zip(_near, _same), key=lambda z: -_best_s[z[0]])[:12]],
}
print(f"Near-duplicate test queries (cosine >= 0.9): {len(_near)} of {len(test)} ({len(_near) / max(len(test), 1):.1%}); different labels: {data_quality['near_duplicates']['pairs_with_different_labels']}")
data_quality["gate"] = [{"check": n, "status": "pass" if ok else ("fail" if h else "warn")} for n, ok, h in _gate]
Path("results").mkdir(exist_ok=True)
Path("results/data_quality.json").write_text(json.dumps(data_quality, indent=2), encoding="utf-8")
if _hard_fail:
    raise RuntimeError("Data-quality gate failed before training: " + "; ".join(_hard_fail))

## 6. Text preprocessing

The preprocessing step is intentionally modest. Lower-casing and whitespace normalisation are applied, while names, numbers, airline codes, city names, and question wording are retained because they may carry useful intent information.

The original query is kept alongside the cleaned version so the exported files remain easy to inspect.

In [ ]:
def preprocess_text(value):
    value = str(value).lower()
    return " ".join(value.split())

train["clean_text"] = train[TEXT_COL].map(preprocess_text)
test["clean_text"] = test[TEXT_COL].map(preprocess_text)

median_length = train["word_count"].median()
train["length_group"] = np.where(train["word_count"] <= median_length, "short", "long")
test["length_group"] = np.where(test["word_count"] <= median_length, "short", "long")

INTERROGATIVE_STARTERS = {
    "what", "which", "where", "when", "how", "can", "could",
    "do", "does", "is", "are", "would", "will"
}

def query_style(value):
    words = value.split()
    return "interrogative" if words and words[0] in INTERROGATIVE_STARTERS else "request_fragment"

train["query_style"] = train["clean_text"].map(query_style)
test["query_style"] = test["clean_text"].map(query_style)

train.to_csv("data/processed/train_processed.csv", index=False)
test.to_csv("data/processed/test_processed.csv", index=False)

print("Example preprocessing:")
display(train[[TEXT_COL, "clean_text"]].head())

## 7. Automatic model search

This is a small, transparent AutoML step rather than a black-box tuning service.

Candidate pipelines:

1. TF-IDF word/word-bigram features + Logistic Regression
2. TF-IDF word/word-bigram features + Linear SVM
3. TF-IDF word/word-bigram features + Multinomial Naive Bayes

Each candidate is evaluated with stratified cross-validation **using the training set only**. Macro-F1 is the selection metric because it gives every intent equal weight.

After selection, the chosen pipeline is refit on all training rows and evaluated once on the held-out test set.

This avoids selecting a model because it happened to perform best on the final test set.

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

X_train = train["clean_text"]
y_train = train[INTENT_COL]
X_test = test["clean_text"]
y_test = test[INTENT_COL]

class_counts = y_train.value_counts()

# Stratified CV needs every class to have at least as many rows as folds.
# Classes that are too small for CV are excluded from MODEL SELECTION ONLY.
# They are still used to fit the final model and still appear in the held-out test evaluation.
effective_folds = int(CV_FOLDS)
cv_eligible_classes = class_counts[class_counts >= effective_folds].index
if len(cv_eligible_classes) < 2:
    effective_folds = 2
    cv_eligible_classes = class_counts[class_counts >= effective_folds].index
if len(cv_eligible_classes) < 2:
    raise ValueError(
        "Model selection needs at least two classes with two or more training examples each. "
        f"Class counts: {class_counts.to_dict()}"
    )

cv_mask = y_train.isin(cv_eligible_classes)
X_cv, y_cv = X_train[cv_mask], y_train[cv_mask]
cv_excluded_classes = sorted(class_counts.index.difference(cv_eligible_classes).astype(str).tolist())
cv_excluded_rows = int((~cv_mask).sum())
if cv_excluded_classes:
    print(
        f"NOTE: {len(cv_excluded_classes)} class(es) with fewer than {effective_folds} training examples "
        f"({cv_excluded_rows} rows) are excluded from CV model selection only: {cv_excluded_classes}"
    )
    print("They are still used to fit the final model and are still evaluated on the held-out test set.")

candidate_models = {
    "TF-IDF + Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
    ]),
    "TF-IDF + Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ("classifier", LinearSVC(class_weight="balanced", max_iter=5000, random_state=RANDOM_SEED)),
    ]),
    "TF-IDF + Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ("classifier", MultinomialNB()),
    ]),
}

cv = StratifiedKFold(n_splits=effective_folds, shuffle=True, random_state=RANDOM_SEED)
search_rows = []

for name, pipeline in candidate_models.items():
    scores = cross_validate(
        pipeline, X_cv, y_cv, cv=cv,
        scoring={"macro_f1": "f1_macro", "accuracy": "accuracy"},
        n_jobs=-1, error_score="raise",
    )
    search_rows.append({
        "Model": name,
        "CV_Macro_F1": round(float(scores["test_macro_f1"].mean()), 4),
        "CV_Macro_F1_SD": round(float(scores["test_macro_f1"].std()), 4),
        "CV_Accuracy": round(float(scores["test_accuracy"].mean()), 4),
        "CV_Accuracy_SD": round(float(scores["test_accuracy"].std()), 4),
    })

model_comparison = pd.DataFrame(search_rows).sort_values(
    ["CV_Macro_F1", "CV_Accuracy"], ascending=False
).reset_index(drop=True)
best_model_name = model_comparison.iloc[0]["Model"]
model_comparison.to_csv("results/model_comparison.csv", index=False)

print(f"Cross-validation folds used: {effective_folds}")
print(f"Selected model from training-only CV: {best_model_name}")
display(model_comparison)


In [ ]:
# The polished model-comparison graph is generated in the final analysis section after the final model is fixed.


## 8. Optional Sentence-BERT benchmark

Sentence-BERT is kept separate from the main AutoML search because it requires downloading a pretrained model and uses a different feature representation.

If `RUN_SBERT = True`, this section adds a semantic benchmark to the exported model comparison. The classical TF-IDF AutoML winner remains the reproducible selection baseline; the SBERT result is reported separately unless you deliberately extend the selection procedure.

In [ ]:
SBERT_OK = False

if RUN_SBERT:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3,<6"], check=True)
        from sentence_transformers import SentenceTransformer

        sbert = SentenceTransformer(SBERT_MODEL_NAME)
        train_embeddings = sbert.encode(X_cv.tolist(), show_progress_bar=True, convert_to_numpy=True)
        sbert_cv_rows = []
        for fold_train, fold_valid in cv.split(train_embeddings, y_cv):
            clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
            clf.fit(train_embeddings[fold_train], y_cv.iloc[fold_train])
            pred = clf.predict(train_embeddings[fold_valid])
            sbert_cv_rows.append({
                "accuracy": accuracy_score(y_cv.iloc[fold_valid], pred),
                "macro_f1": precision_recall_fscore_support(y_cv.iloc[fold_valid], pred, average="macro", zero_division=0)[2],
            })
        sbert_cv_df = pd.DataFrame(sbert_cv_rows)
        sbert_row = pd.DataFrame([{
            "Model": "SBERT + Logistic Regression",
            "CV_Macro_F1": round(float(sbert_cv_df["macro_f1"].mean()), 4),
            "CV_Macro_F1_SD": round(float(sbert_cv_df["macro_f1"].std()), 4),
            "CV_Accuracy": round(float(sbert_cv_df["accuracy"].mean()), 4),
            "CV_Accuracy_SD": round(float(sbert_cv_df["accuracy"].std()), 4),
        }])
        sbert_row.to_csv("results/sbert_benchmark.csv", index=False)
        SBERT_OK = True
        print("SBERT training-only CV benchmark completed. It is reported separately and is not used for final model selection.")
        display(sbert_row)
    except Exception as exc:
        print("SBERT benchmark was skipped; the core TF-IDF pipeline remains complete.")
        print(f"Reason: {type(exc).__name__}: {str(exc)[:300]}")
else:
    print("SBERT is disabled. Set RUN_SBERT = True in the configuration cell to enable it.")


## 9. Final model evaluation

The selected model was chosen before looking at the held-out test labels.

Now it is evaluated on the test set. These are the numbers that should be presented as the final test results.

In [ ]:

# Fit the selected model once on the full training set, then evaluate the held-out test set once.
best_model = candidate_models[best_model_name]
best_model.fit(X_train, y_train)
best_predictions = best_model.predict(X_test)

# Keep the selected model available for live prediction and explainability.
fitted = {best_model_name: best_model}
predictions = {best_model_name: best_predictions}

test_accuracy = accuracy_score(y_test, best_predictions)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test, best_predictions, average="macro", zero_division=0
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_test, best_predictions, average="weighted", zero_division=0
)

final_metrics = {
    "final_model": best_model_name,
    "selection_metric": "cross_validation_macro_f1",
    "selection_cv_folds": effective_folds,
    "selection_cv_excluded_classes": cv_excluded_classes,
    "selection_cv_excluded_rows": cv_excluded_rows,
    "accuracy": round(float(test_accuracy), 4),
    "macro_precision": round(float(macro_precision), 4),
    "macro_recall": round(float(macro_recall), 4),
    "macro_f1": round(float(macro_f1), 4),
    "weighted_precision": round(float(weighted_precision), 4),
    "weighted_recall": round(float(weighted_recall), 4),
    "weighted_f1": round(float(weighted_f1), 4),
    "test_only_intents": test_only_intents,
}
Path("results/final_metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
Path("results/final_model_metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
print(json.dumps(final_metrics, indent=2))


In [ ]:
labels = sorted(y_test.unique())
report = classification_report(
    y_test, best_predictions, labels=labels, zero_division=0, output_dict=True
)

per_intent = pd.DataFrame(report).transpose()
per_intent.to_csv("results/per_intent_performance.csv")

plt.figure(figsize=(9, max(5, len(per_intent) * 0.25)))
intent_rows = per_intent.drop(
    index=["accuracy", "macro avg", "weighted avg"], errors="ignore"
).sort_values("f1-score")

plt.barh(intent_rows.index, intent_rows["f1-score"], color=STUDIO_COLORS["accent"])
plt.xlabel("F1-score")
plt.title(f"Per-intent performance — {best_model_name}")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig("results/figures/per_intent_f1.png", dpi=150, bbox_inches="tight")
plt.show()

display(per_intent)

In [ ]:
cm = confusion_matrix(y_test, best_predictions, labels=labels)

fig_width = max(8, min(16, len(labels) * 0.45))
fig_height = max(6, min(14, len(labels) * 0.45))

plt.figure(figsize=(fig_width, fig_height))
from matplotlib.colors import PowerNorm
plt.imshow(cm, cmap=STUDIO_CMAP, norm=PowerNorm(gamma=0.35))   # compressed scale so small error counts stay visible next to the big cells
plt.grid(False)
plt.colorbar()
plt.title(f"Confusion matrix — {best_model_name}")
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.tight_layout()
plt.savefig("results/figures/confusion_matrix_best_model.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Error analysis

The error file is intentionally row-level. It is useful during the project presentation because you can inspect real examples rather than discussing the score alone.

The notebook does not invent explanations for an incorrect prediction. If you want a qualitative explanation, add it manually in the exported CSV.

In [ ]:
error_mask = best_predictions != y_test.to_numpy()

error_df = pd.DataFrame({
    "Query": test.loc[error_mask, TEXT_COL].to_numpy(),
    "Actual": y_test.to_numpy()[error_mask],
    "Predicted": best_predictions[error_mask],
    "Source_Dataset": test.loc[error_mask, "source_dataset"].to_numpy(),
})

error_sample = error_df.head(20).copy() if len(error_df) else error_df.copy()

error_path = Path("results/error_analysis_sample.csv")
error_sample.to_csv(error_path, index=False)

print(f"Incorrect test predictions: {len(error_df):,}")
display(error_sample)

## 11. Textual subgroup audit

This section compares performance across query length, question/request style, and source dataset.

These are **textual and dataset subgroups**, not demographic fairness groups.

In [ ]:
audit_rows = []

for column, category_name in [
    ("length_group", "Length"),
    ("query_style", "Style"),
    ("source_dataset", "Source Dataset"),
]:
    for value in sorted(test[column].dropna().unique()):
        mask = (test[column] == value).to_numpy()
        if not mask.any():
            continue

        subgroup_accuracy = accuracy_score(y_test.to_numpy()[mask], best_predictions[mask])
        _, _, subgroup_f1, _ = precision_recall_fscore_support(
            y_test.to_numpy()[mask],
            best_predictions[mask],
            average="macro",
            zero_division=0,
        )

        audit_rows.append({
            "Category": category_name,
            "Subgroup": value,
            "Samples": int(mask.sum()),
            "Accuracy": round(float(subgroup_accuracy), 4),
            "Macro_F1": round(float(subgroup_f1), 4),
        })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv("results/audit_results.csv", index=False)

display(audit_df)

## 12. Feature-level explanation

For the TF-IDF models, the classifier coefficients provide a useful inspection of which words or phrases are associated with each intent.

This is an association-based explanation, not a causal explanation.

If the final model is not a TF-IDF pipeline, this section reports that word-level coefficients are not directly available.

In [ ]:
if best_model_name in fitted:
    tfidf = fitted[best_model_name].named_steps["tfidf"]
    classifier = fitted[best_model_name].named_steps["classifier"]
    feature_names = np.asarray(tfidf.get_feature_names_out())

    def class_coefficients(model, class_index):
        coef = np.asarray(model.coef_)
        if coef.shape[0] == 1 and len(model.classes_) == 2:
            return coef[0] if class_index == 1 else -coef[0]
        return coef[class_index]

    top_features = {}
    for class_index, class_name in enumerate(classifier.classes_):
        coefficients = class_coefficients(classifier, class_index)
        top_indices = np.argsort(coefficients)[-10:][::-1]
        top_features[str(class_name)] = feature_names[top_indices].tolist()

    Path("results/top_features_per_intent.json").write_text(
        json.dumps(top_features, indent=2), encoding="utf-8"
    )

    for class_name, features in list(top_features.items())[:5]:
        print(f"{class_name}: {', '.join(features)}")
else:
    print("Word-level coefficients are not available for the selected model.")


## 13. Try your own travel query

This section lets you test the model with a query you choose. Nothing is sampled randomly: the text entered below is sent through the same preprocessing function and the fitted model selected during the experiment.

Try examples such as:

- `I need to change my flight to tomorrow`
- `How many bags can I take on the flight?`
- `What time does my flight leave?`
- `I want to book a flight from Delhi to Mumbai`
- `Can I cancel my reservation?`

The predicted intent comes from the model trained in this notebook. If the classifier supports probabilities, the result also shows the model's top scores.


In [ ]:
def predict_query(text, top_k_features=5):
    """Predict the intent for one user-entered travel query."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please enter a non-empty travel query.")

    if best_model_name not in fitted:
        raise RuntimeError("The selected model is not available. Run the model-selection cells first.")

    cleaned = preprocess_text(text)
    pipeline = fitted[best_model_name]
    prediction = pipeline.predict([cleaned])[0]

    print(f"Model used: {best_model_name}")
    print(f"Query: {text}")
    print(f"Predicted intent: {prediction}")

    # Show confidence-like information only when the fitted classifier provides it.
    classifier = pipeline.named_steps.get("classifier") if hasattr(pipeline, "named_steps") else None
    if classifier is not None and hasattr(pipeline, "predict_proba"):
        probabilities = pipeline.predict_proba([cleaned])[0]
        classes = pipeline.classes_
        top_indices = np.argsort(probabilities)[::-1][:3]
        print("Top model probabilities:")
        for index in top_indices:
            print(f"  {classes[index]}: {probabilities[index]:.3f}")
    elif classifier is not None and hasattr(pipeline, "decision_function"):
        scores = pipeline.decision_function([cleaned])
        classes = pipeline.classes_
        scores = np.atleast_1d(scores)
        if scores.ndim > 1:
            scores = scores[0]
        top_indices = np.argsort(scores)[::-1][:3]
        print("Top decision scores (not probabilities):")
        for index in top_indices:
            print(f"  {classes[index]}: {scores[index]:.3f}")
    else:
        print("Confidence scores are not available for the selected model.")

    # For linear TF-IDF classifiers, show a small, model-based term inspection.
    if (hasattr(pipeline, "named_steps") and
        "tfidf" in pipeline.named_steps and
        "classifier" in pipeline.named_steps and
        hasattr(pipeline.named_steps["classifier"], "coef_")):
        tfidf = pipeline.named_steps["tfidf"]
        classifier = pipeline.named_steps["classifier"]
        vector = tfidf.transform([cleaned])
        feature_names = np.asarray(tfidf.get_feature_names_out())
        class_index = list(classifier.classes_).index(prediction)
        coefficients = np.asarray(classifier.coef_)
        if coefficients.shape[0] == 1 and len(classifier.classes_) == 2:
            coefficients = coefficients[0] if class_index == 1 else -coefficients[0]
        else:
            coefficients = coefficients[class_index]
        contributions = vector.multiply(coefficients).toarray()[0]
        positive_indices = np.argsort(contributions)[-top_k_features:][::-1]
        terms = [feature_names[i] for i in positive_indices if contributions[i] > 0]
        if terms:
            print("Terms supporting the predicted class:", ", ".join(terms))

    return prediction

# Enter your own query. This is deliberately not a random/sample prediction.
# An empty entry (or a non-interactive "Run all") skips the demo instead of stopping the notebook,
# so the export cells below can still run. Re-run this cell any time to try another query.
try:
    user_query = input("Enter a travel query (press Enter to skip): ").strip()
except EOFError:
    user_query = ""

if user_query:
    predict_query(user_query)
else:
    print("No query entered; live prediction skipped. Re-run this cell to try your own query.")


## 13b. Autocomplete / next-word prediction

Type the beginning of a travel query and the dashboard suggests the most likely next words, finishes a half-typed word, proposes a few complete queries and a likely intent. It is a **word n-gram language model with stupid-backoff** built from the **training queries only** (the same train split used above, whether supplied or created by the notebook) and scored on the held-out test queries.

**Guarding against over- and underfitting**
- Settings (context length, minimum count, backoff weight) are chosen on a **validation slice taken from the training data**. The test set is never used for tuning; ties go to the simpler model.
- A **5-fold cross-validation** on the training data shows how stable the score is.
- The final model is scored **once** on the test queries, next to a word-frequency baseline and a unigram model. A large train-vs-test gap flags overfitting; barely beating the unigram model flags underfitting.

Exported as `results/next_word_model.json`; the dashboard only looks counts up and never trains. It only knows phrasing seen in this dataset.

In [ ]:
import math
import re
from collections import Counter, defaultdict

NW_TOKEN = re.compile(r"[a-z0-9_']+")
NW_FORMAT = "travel_nlp_ngram_v2"
NW_MAX_TOKENS = 60
NW_TOP_NEXT = 8
NW_GRID = {"max_context": [1, 2, 3, 4], "min_count": [1, 2, 3], "alpha": [0.2, 0.4, 0.6]}
NW_DEFAULT = {"max_context": 2, "min_count": 2, "alpha": 0.4}


def nw_tokens(text):
    return NW_TOKEN.findall(str(text).lower())[:NW_MAX_TOKENS]


def build_next_word_model(texts, intents, max_context=2, min_count=2, alpha=0.4):
    contexts, prior = defaultdict(Counter), Counter()
    token_intent, intent_tokens, word_counts = defaultdict(Counter), Counter(), Counter()
    n_queries = 0
    for text, intent in zip(texts, intents):
        toks = nw_tokens(text)
        if not toks:
            continue
        n_queries += 1
        prior[str(intent)] += 1
        for tok in toks:
            token_intent[tok][str(intent)] += 1
            intent_tokens[str(intent)] += 1
            word_counts[tok] += 1
        seq = ["<s>"] + toks + ["</s>"]
        for i in range(1, len(seq)):
            for n in range(0, max_context + 1):
                if n > i:
                    continue
                contexts[" ".join(seq[i - n:i]) if n else ""][seq[i]] += 1
    packed = {}
    for key, counter in contexts.items():
        total = sum(counter.values())
        if key and total < min_count:
            continue
        top = counter.most_common(60 if key == "" else NW_TOP_NEXT)
        packed[key] = [total, {w: c for w, c in top}]
    return {
        "format": NW_FORMAT,
        "trained_on": "training split only",
        "max_context_tokens": max_context,
        "min_context_count": min_count,
        "backoff_alpha": alpha,
        "n_train_queries": n_queries,
        "vocab_size": len(word_counts),
        "contexts": packed,
        "word_counts": {w: c for w, c in word_counts.items() if c >= min_count},
        "intent_prior": dict(prior),
        "intent_tokens": dict(intent_tokens),
        "token_intent": {t: dict(c) for t, c in token_intent.items() if word_counts[t] >= min_count},
    }


def next_word_suggestions(model, prefix_tokens, k=5):
    """Stupid-backoff ranking. Returns [(word, score)] best first; '</s>' means 'end of query'."""
    max_ctx = int(model["max_context_tokens"])
    seq = ["<s>"] + list(prefix_tokens)[-max_ctx:]
    scores = {}
    for level, n in enumerate(range(max_ctx, -1, -1)):
        if n > len(seq):
            continue
        entry = model["contexts"].get(" ".join(seq[-n:]) if n else "")
        if not entry:
            continue
        total, nexts = entry
        for word, count in nexts.items():
            if word != "<s>" and word not in scores:
                scores[word] = (count / total) * (float(model["backoff_alpha"]) ** level)
    return sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))[:k]


def prefix_intent_hint(model, prefix_tokens, k=3):
    prior, vocab = model["intent_prior"], max(int(model["vocab_size"]), 1)
    total_q = sum(prior.values())
    logp = {}
    for intent, count in prior.items():
        lp = math.log(count / total_q)
        denom = model["intent_tokens"].get(intent, 0) + vocab
        for tok in prefix_tokens:
            if tok in model["token_intent"]:
                lp += math.log((model["token_intent"][tok].get(intent, 0) + 1) / denom)
        logp[intent] = lp
    peak = max(logp.values())
    weights = {i: math.exp(v - peak) for i, v in logp.items()}
    norm = sum(weights.values())
    return sorted(((i, w / norm) for i, w in weights.items()), key=lambda kv: -kv[1])[:k]


def nw_hit_rates(model, texts, ks=(1, 3, 5), max_queries=None):
    """Share of word positions whose true next word is in the model's top-k. '</s>' counts as a target."""
    hits = {k: 0 for k in ks}
    positions = 0
    for n, text in enumerate(texts):
        if max_queries is not None and n >= max_queries:
            break
        toks = nw_tokens(text)
        if not toks:
            continue
        for i, target in enumerate(toks + ["</s>"]):
            ranked = [w for w, _ in next_word_suggestions(model, toks[:i], k=max(ks))]
            positions += 1
            for k in ks:
                hits[k] += target in ranked[:k]
    return {f"top{k}": (hits[k] / positions if positions else None) for k in ks} | {"positions": positions}


# ---------------------------------------------------------------------------
# 1) Choose settings on a VALIDATION slice carved out of TRAIN. The test set is not used here.
# ---------------------------------------------------------------------------
nw_texts = train[TEXT_COL].astype(str).reset_index(drop=True)
nw_labels = train[INTENT_COL].astype(str).reset_index(drop=True)
nw_rng = np.random.default_rng(RANDOM_SEED)
nw_order = nw_rng.permutation(len(nw_texts))
n_val = int(round(0.15 * len(nw_texts)))
nw_tuned = n_val >= 30 and (len(nw_texts) - n_val) >= 100
tuning_rows = []
selected = dict(NW_DEFAULT)
if nw_tuned:
    val_idx, sub_idx = nw_order[:n_val], nw_order[n_val:]
    val_texts = nw_texts.iloc[val_idx]
    for mc in NW_GRID["max_context"]:
        for mn in NW_GRID["min_count"]:
            for al in NW_GRID["alpha"]:
                m_ = build_next_word_model(nw_texts.iloc[sub_idx], nw_labels.iloc[sub_idx], mc, mn, al)
                r_ = nw_hit_rates(m_, val_texts)
                tuning_rows.append({"max_context": mc, "min_count": mn, "alpha": al,
                                    "val_top1": r_["top1"], "val_top3": r_["top3"], "val_top5": r_["top5"]})
    # Best validation top-3; ties go to the SIMPLER model (shorter context, higher min_count) to resist overfitting.
    best_row = sorted(tuning_rows, key=lambda r: (-round(r["val_top3"], 4), r["max_context"], -r["min_count"], r["alpha"]))[0]
    selected = {k: best_row[k] for k in ("max_context", "min_count", "alpha")}

# ---------------------------------------------------------------------------
# 2) Stability check: 5-fold cross-validation on TRAIN with the chosen settings.
# ---------------------------------------------------------------------------
cv_scores = []
if len(nw_texts) >= 50:
    folds = np.array_split(nw_order, 5)
    for f in range(5):
        hold = folds[f]
        rest = np.concatenate([folds[j] for j in range(5) if j != f])
        m_ = build_next_word_model(nw_texts.iloc[rest], nw_labels.iloc[rest], **selected)
        cv_scores.append(nw_hit_rates(m_, nw_texts.iloc[hold])["top3"])

# ---------------------------------------------------------------------------
# 3) Final model: all training queries, chosen settings. Scored ONCE on the held-out test queries.
# ---------------------------------------------------------------------------
next_word_model = build_next_word_model(nw_texts, nw_labels, **selected)
test_rates = nw_hit_rates(next_word_model, test[TEXT_COL])
train_rates = nw_hit_rates(next_word_model, nw_texts, max_queries=1500)   # in-sample, only to measure the gap

fixed_top5 = [w for w, _ in sorted(next_word_model["contexts"][""][1].items(), key=lambda kv: -kv[1])[:5]]
vocab_known = set(next_word_model["word_counts"]) | {"</s>"}
base_hits, unseen, positions = {1: 0, 3: 0, 5: 0}, 0, 0
for text in test[TEXT_COL]:
    toks = nw_tokens(text)
    for target in (toks + ["</s>"] if toks else []):
        positions += 1
        unseen += target not in vocab_known
        for k_ in base_hits:
            base_hits[k_] += target in fixed_top5[:k_]
uni = build_next_word_model(nw_texts, nw_labels, max_context=1, min_count=selected["min_count"], alpha=selected["alpha"])
uni["max_context_tokens"] = 0   # unigram only: ranks by overall word frequency
uni_rates = nw_hit_rates(uni, test[TEXT_COL])

# Intent hint evaluated at the halfway point of test queries with at least 4 words.
train_majority = nw_labels.value_counts().idxmax()
hint_n = hint_top1 = hint_top3 = base_top1 = 0
for text, intent in zip(test[TEXT_COL], test[INTENT_COL]):
    toks = nw_tokens(text)
    if len(toks) < 4:
        continue
    hint = [i for i, _ in prefix_intent_hint(next_word_model, toks[: (len(toks) + 1) // 2], k=3)]
    hint_n += 1
    hint_top1 += hint[:1] == [str(intent)]
    hint_top3 += str(intent) in hint
    base_top1 += str(intent) == train_majority

gap = train_rates["top3"] - test_rates["top3"]            # informational only: an n-gram model memorises its own counts
cv_mean = float(np.mean(cv_scores)) if cv_scores else None
cv_sd = float(np.std(cv_scores, ddof=1)) if len(cv_scores) > 1 else None
shift = (cv_mean - test_rates["top3"]) if cv_mean is not None else None   # held-out-in-train vs real test set

# Sensitivity (reporting only, never used to choose): does the tuned model beat a simpler fixed one on the test queries?
simple_model = build_next_word_model(nw_texts, nw_labels, **NW_DEFAULT)
simple_rates = nw_hit_rates(simple_model, test[TEXT_COL])

if test_rates["top3"] < uni_rates["top3"] + 0.05:
    fit_verdict = "possible underfitting: barely better than a word-frequency model"
elif cv_mean is None:
    fit_verdict = "not checked: too few training queries for a reliable over/underfitting check; treat the scores as indicative only"
elif shift is not None and shift > 0.10:
    fit_verdict = "possible overfitting: the cross-validation score is much higher than the test score"
elif test_rates["top3"] + 0.005 < simple_rates["top3"]:
    fit_verdict = "possible tuning overfit: a simpler fixed model scores higher on the test queries"
elif cv_sd is not None and cv_sd > 0.03:
    fit_verdict = "no sign of over- or underfitting, but scores vary between folds (small training set), so treat them as indicative"
elif shift is not None and shift > 0.05:
    fit_verdict = "no sign of overfitting; test queries are somewhat harder than training folds (distribution shift), not a model fault"
else:
    fit_verdict = "no sign of over- or underfitting on these checks"

next_word_model["selected_settings"] = selected
next_word_model["settings_chosen_by"] = (
    "grid search on a 15% validation slice of the training queries (test set not used)" if nw_tuned
    else "default settings: too few training queries to hold out a validation slice")
next_word_model["tuning_grid"] = tuning_rows
next_word_model["evaluation"] = {
    "method": "every word position of every held-out test query; the end-of-query marker counts as a target",
    "test_queries": int(len(test)),
    "positions": int(test_rates["positions"]),
    "top1_accuracy": test_rates["top1"], "top3_accuracy": test_rates["top3"], "top5_accuracy": test_rates["top5"],
    "frequent_words_baseline_top1": base_hits[1] / positions, "frequent_words_baseline_top3": base_hits[3] / positions,
    "frequent_words_baseline_top5": base_hits[5] / positions,
    "unigram_model_top3": uni_rates["top3"],
    "unseen_target_word_rate": unseen / positions,
    "train_in_sample_top3": train_rates["top3"],
    "train_minus_test_top3_gap": gap,
    "cv_minus_test_top3_shift": shift,
    "simpler_default_model_test_top3": simple_rates["top3"],
    "cv_top3_mean": cv_mean, "cv_top3_sd": cv_sd, "cv_folds": len(cv_scores),
    "fit_verdict": fit_verdict,
    "intent_hint_queries": int(hint_n),
    "intent_hint_top1_at_half_query": (hint_top1 / hint_n) if hint_n else None,
    "intent_hint_top3_at_half_query": (hint_top3 / hint_n) if hint_n else None,
    "intent_hint_majority_baseline_top1": (base_top1 / hint_n) if hint_n else None,
}
next_word_model["limitations"] = [
    "Only knows wording seen in the training queries of this dataset.",
    "Unseen words get no suggestion of their own; the model backs off to the previous words.",
    "The intent hint is Naive Bayes over word counts and is not the classifier selected above.",
    "The in-sample score is optimistic by construction; the held-out test and cross-validation scores are the honest ones.",
]
nw_path = Path("results/next_word_model.json")
nw_path.write_text(json.dumps(next_word_model, separators=(",", ":"), ensure_ascii=False), encoding="utf-8")
if nw_path.stat().st_size > 12 * 1024 * 1024:
    raise ValueError("next_word_model.json is unexpectedly large; raise NW_GRID min_count values.")

ev = next_word_model["evaluation"]
print(f"Autocomplete model: {len(next_word_model['contexts']):,} contexts, {next_word_model['vocab_size']:,} words, {nw_path.stat().st_size / 1024:.0f} KB")
print(f"Settings: {selected} ({next_word_model['settings_chosen_by']})")
print(f"Held-out test, {ev['positions']:,} word positions: top-1 {ev['top1_accuracy']:.1%}, top-3 {ev['top3_accuracy']:.1%}, top-5 {ev['top5_accuracy']:.1%}")
print(f"Word-frequency baseline top-3 {ev['frequent_words_baseline_top3']:.1%}; unigram model top-3 {ev['unigram_model_top3']:.1%}")
if cv_mean is not None:
    print(f"5-fold CV top-3 {cv_mean:.1%} +/- {cv_sd:.1%} | test top-3 {ev['top3_accuracy']:.1%} | simpler fixed model test top-3 {simple_rates['top3']:.1%} | (in-sample {ev['train_in_sample_top3']:.1%}, informational)")
print("Fit check:", fit_verdict)
demo_prefix = nw_tokens(test[TEXT_COL].iloc[0])[:2]
print(f"\nExample prefix: {' '.join(demo_prefix)!r}")
print("Likely next words:", [(w, round(s, 3)) for w, s in next_word_suggestions(next_word_model, demo_prefix, k=5)])
print("Likely intents:  ", [(i, round(p, 3)) for i, p in prefix_intent_hint(next_word_model, demo_prefix, k=3)])


## 13c. Portable classifier export (for the Studio Query Lab)

The selected model is also saved as **plain JSON** (`results/classifier_model.json`): the TF-IDF vocabulary and weights plus the classifier coefficients. JSON is used on purpose instead of a pickle/joblib file, because a pickle can execute code when loaded and the Studio opens ZIP files from other people.

The cell re-implements prediction from the JSON alone and checks that it reproduces scikit-learn's predictions on the **entire held-out test set**. If they disagree, the model is not exported and the Studio will not offer live classification for this run.

In [ ]:
import math, re as _re
from collections import Counter as _Counter

CLF_FORMAT = "travel_nlp_linear_v1"
_TOKEN = _re.compile(r"(?u)\b\w\w+\b")

def portable_scores(model, text):
    """Score one query from the exported JSON only (no scikit-learn)."""
    toks = _TOKEN.findall(preprocess_text(text))
    lo, hi = model["ngram_range"]
    terms = []
    for n in range(lo, hi + 1):
        terms += [" ".join(toks[k:k + n]) for k in range(len(toks) - n + 1)]
    counts = _Counter(t for t in terms if t in model["vocab"])
    feats = {}
    for t, c in counts.items():
        tf = 1.0 + math.log(c) if model["sublinear_tf"] else float(c)
        feats[model["vocab"][t]] = tf * model["idf"][model["vocab"][t]]
    norm = math.sqrt(sum(v * v for v in feats.values()))
    if norm > 0:
        feats = {k: v / norm for k, v in feats.items()}
    out = []
    for row, b in zip(model["weights"], model["bias"]):
        out.append(b + sum(row[k] * v for k, v in feats.items()))
    return out, len(counts)

def portable_predict(model, text):
    s, known = portable_scores(model, text)
    order = sorted(range(len(s)), key=lambda k: -s[k])
    return model["classes"][order[0]], (s[order[0]] - s[order[1]]) if len(order) > 1 else 0.0, known

portable_parity_ok = None   # None = not applicable
portable_parity = {"status": "skipped", "reason": "unsupported model type"}
try:
    _tf = best_model.named_steps["tfidf"]; _cl = best_model.named_steps["classifier"]
    if _tf.analyzer != "word" or not _tf.lowercase or _tf.use_idf is not True or _tf.norm != "l2" or _tf.token_pattern != r"(?u)\b\w\w+\b" or _tf.stop_words or _tf.max_features:
        raise TypeError("non-default vectorizer settings")
    if isinstance(_cl, MultinomialNB):
        _W, _b, _kind = np.asarray(_cl.feature_log_prob_), np.asarray(_cl.class_log_prior_), "naive_bayes"
    elif hasattr(_cl, "coef_"):
        _W, _b, _kind = np.asarray(_cl.coef_), np.asarray(_cl.intercept_), ("linear_svm" if isinstance(_cl, LinearSVC) else "logistic_regression")
        if _W.shape[0] == 1:                      # binary problems store one row: expand to two classes
            _W, _b = np.vstack([-_W[0], _W[0]]), np.array([-_b[0], _b[0]])
    else:
        raise TypeError("unsupported classifier")
    _vocab = {t: int(i) for t, i in _tf.vocabulary_.items()}
    classifier_model = {
        "format": CLF_FORMAT, "kind": _kind, "model_name": best_model_name,
        "classes": [str(c) for c in _cl.classes_],
        "ngram_range": [int(_tf.ngram_range[0]), int(_tf.ngram_range[1])], "sublinear_tf": bool(_tf.sublinear_tf),
        "vocab": _vocab, "idf": [round(float(x), 6) for x in _tf.idf_],
        "weights": [[round(float(x), 6) for x in row] for row in _W], "bias": [round(float(x), 6) for x in _b],
        "preprocessing": "lowercase + whitespace normalisation (same as the notebook)",
        "score_note": "Scores are decision values, not probabilities. Margin = best score minus second-best score.",
        "trained_on": {"rows": int(len(train)), "intents": int(train[INTENT_COL].nunique()), "random_seed": int(RANDOM_SEED)},
    }
    _mine = [portable_predict(classifier_model, t)[0] for t in test[TEXT_COL]]
    _diff = int(sum(a != b for a, b in zip(_mine, best_predictions)))
    portable_parity = {"status": "pass" if _diff == 0 else "fail", "test_queries_checked": int(len(_mine)), "disagreements": _diff}
    portable_parity_ok = (_diff == 0)
    if portable_parity_ok:
        _p = Path("results/classifier_model.json")
        _p.write_text(json.dumps({**classifier_model, "parity": portable_parity}, separators=(",", ":")), encoding="utf-8")
        print(f"Exported {_p} ({_p.stat().st_size / 1024:.0f} KB, {len(_vocab):,} features, {len(classifier_model['classes'])} intents)")
    else:
        print(f"NOT exported: portable predictions disagree with scikit-learn on {_diff} test queries.")
except Exception as _exc:
    print(f"Portable classifier not exported ({_exc}). The Studio will show stored results only for this run.")
print("Parity with scikit-learn:", portable_parity)

## 14. Final analysis, graphs, and quality checks

This section adds the final audit and presentation layer before the dashboard bundle is created. All figures and tables are generated from the current run; no demonstration metrics are hard-coded. The checks are deliberately conservative so that a broken or misleading experiment is stopped before export.


In [ ]:
# Final audit and presentation artifacts
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

FIG_DIR = Path("results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Baseline is contextual only and never used for model selection.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_accuracy = float(accuracy_score(y_test, baseline_pred))
baseline_macro_f1 = float(f1_score(y_test, baseline_pred, average="macro", zero_division=0))

comparison_plus = pd.concat([
    model_comparison.copy(),
    pd.DataFrame([{
        "Model": "Majority-class baseline",
        "CV_Accuracy": np.nan,
        "CV_Accuracy_SD": np.nan,
        "CV_Macro_F1": np.nan,
        "CV_Macro_F1_SD": np.nan,
        "Test_Accuracy": baseline_accuracy,
        "Test_Macro_F1": baseline_macro_f1,
    }])
], ignore_index=True)
comparison_plus.to_csv("results/model_comparison_with_baseline.csv", index=False)

# Data-quality chart: compact and easy to interpret.
quality_counts = pd.Series({
    "Missing text": int(train[TEXT_COL].isna().sum()),
    "Missing labels": int(train[INTENT_COL].isna().sum()),
    "Blank text": int((train[TEXT_COL].astype(str).str.strip() == "").sum()),
    "Duplicate text": int(train[TEXT_COL].duplicated().sum()),
    "Conflicting labels": int(len(conflicts)),
})
plt.figure(figsize=(9, 5))
plt.bar(quality_counts.index, quality_counts.values, color=STUDIO_COLORS["accent"])
plt.ylim(0, max(1, float(quality_counts.max())) * 1.25)
for _x, _v in enumerate(quality_counts.values):
    plt.text(_x, _v, f"{int(_v):,}", ha="center", va="bottom", color=STUDIO_COLORS["ink"])
plt.ylabel("Count")
plt.title("Training data-quality checks")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "data_quality_checks.png", dpi=180, bbox_inches="tight")
plt.show()

# Model comparison with CV variability.
model_plot = model_comparison.dropna(subset=["CV_Macro_F1"]).sort_values("CV_Macro_F1")
plt.figure(figsize=(10, max(5, len(model_plot) * 0.7)))
plt.barh(model_plot["Model"], model_plot["CV_Macro_F1"], xerr=model_plot["CV_Macro_F1_SD"], capsize=4, color=STUDIO_COLORS["accent"], ecolor=STUDIO_COLORS["muted"])
plt.xlabel("Cross-validation Macro-F1")
plt.ylabel("Model")
plt.title("Model comparison — training-only CV Macro-F1 ± SD")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(FIG_DIR / "model_comparison_cv_macro_f1.png", dpi=180, bbox_inches="tight")
plt.show()

# Final model versus a simple baseline.
plt.figure(figsize=(9, 5))
plt.bar(["Majority baseline", best_model_name], [baseline_macro_f1, float(final_metrics["macro_f1"])], color=[STUDIO_COLORS["muted"], STUDIO_COLORS["right"]])
plt.ylabel("Macro-F1")
plt.title("Final model vs majority-class baseline")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "baseline_vs_final_model.png", dpi=180, bbox_inches="tight")
plt.show()

# Normalized confusion matrix.
labels = sorted(y_test.unique())
cm_normalized = confusion_matrix(y_test, best_predictions, labels=labels, normalize="true")
plt.figure(figsize=(fig_width, fig_height))
plt.imshow(cm_normalized, vmin=0, vmax=1, cmap=STUDIO_CMAP)
plt.grid(False)
plt.colorbar(label="Share of actual class")
plt.title(f"Normalized confusion matrix — {best_model_name}")
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix_normalized.png", dpi=180, bbox_inches="tight")
plt.show()

# Most common confusion pairs.
pair_counts = pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": best_predictions})
pair_counts = pair_counts[pair_counts["Actual"] != pair_counts["Predicted"]]
pair_counts = pair_counts.groupby(["Actual", "Predicted"]).size().reset_index(name="Errors")
pair_counts = pair_counts.sort_values("Errors", ascending=False).reset_index(drop=True)
pair_counts.to_csv("results/confusion_pairs.csv", index=False)
if len(pair_counts):
    top_pairs = pair_counts.head(15).copy()
    top_pairs["Pair"] = top_pairs["Actual"].astype(str) + " → " + top_pairs["Predicted"].astype(str)
    top_pairs = top_pairs.sort_values("Errors")
    plt.figure(figsize=(10, max(5, len(top_pairs) * 0.35)))
    plt.barh(top_pairs["Pair"], top_pairs["Errors"], color=STUDIO_COLORS["wrong"])
    plt.xlabel("Misclassified test queries")
    plt.ylabel("Actual → predicted")
    plt.title("Most common confusion pairs")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "top_confusion_pairs.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    # No errors on the test set: save a valid image that says so, never an empty/corrupt PNG.
    plt.figure(figsize=(8, 2.2))
    plt.axis("off")
    plt.text(0.5, 0.5, "No misclassified test queries,\nso there are no confusion pairs to plot.",
             ha="center", va="center", fontsize=12)
    plt.savefig(FIG_DIR / "top_confusion_pairs.png", dpi=150, bbox_inches="tight")
    plt.close()

# Top words / n-grams by intent. This is descriptive, not a model explanation.
try:
    ngram_vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
    matrix = ngram_vectorizer.fit_transform(train["clean_text"])
    terms = np.asarray(ngram_vectorizer.get_feature_names_out())
    rows = []
    for intent in train[INTENT_COL].drop_duplicates().tolist():
        mask = train[INTENT_COL].to_numpy() == intent
        scores = np.asarray(matrix[mask].sum(axis=0)).ravel()
        for idx in np.argsort(scores)[-10:][::-1]:
            if scores[idx] > 0:
                rows.append({"Intent": intent, "Term": terms[idx], "Count": int(scores[idx])})
    pd.DataFrame(rows).to_csv("results/top_ngrams_by_intent.csv", index=False)
except Exception as exc:
    Path("results/top_ngrams_error.txt").write_text(str(exc), encoding="utf-8")

# Leakage audit after the final split.
train_keys_final = set(text_key(train[TEXT_COL]))
test_keys_final = set(text_key(test[TEXT_COL]))
final_overlap = train_keys_final & test_keys_final
leakage_audit = pd.DataFrame([
    {"Check": "Final normalized train/test overlap", "Count": len(final_overlap), "Status": "PASS" if len(final_overlap) == 0 else "FAIL"},
    {"Check": "Pre-split normalized train/test overlap", "Count": len(pre_split_overlap), "Status": "PASS" if len(pre_split_overlap) == 0 else "FAIL"},
    {"Check": "Conflicting normalized labels", "Count": len(conflicts), "Status": "PASS" if len(conflicts) == 0 else "FAIL"},
    {"Check": "Duplicate training queries retained", "Count": int(train[TEXT_COL].duplicated().sum()), "Status": "REVIEW" if train[TEXT_COL].duplicated().any() else "PASS"},
])
leakage_audit.to_csv("results/leakage_audit.csv", index=False)

# Prediction uncertainty: probabilities where supported, otherwise decision margins.
if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(X_test)
    max_prob = probabilities.max(axis=1)
    pd.DataFrame({"max_probability": max_prob, "correct": best_predictions == y_test.to_numpy()}).to_csv("results/prediction_uncertainty.csv", index=False)
    plt.figure(figsize=(9, 5))
    plt.hist(max_prob, bins=20, edgecolor="white", color=STUDIO_COLORS["accent"])
    plt.xlabel("Maximum predicted probability")
    plt.ylabel("Test queries")
    plt.title("Prediction probability distribution")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "prediction_probability_distribution.png", dpi=180, bbox_inches="tight")
    plt.show()
elif hasattr(best_model, "decision_function"):
    decision = np.asarray(best_model.decision_function(X_test))
    if decision.ndim == 1:
        margin = np.abs(decision)
    else:
        sorted_scores = np.sort(decision, axis=1)
        margin = sorted_scores[:, -1] - sorted_scores[:, -2] if decision.shape[1] > 1 else np.abs(decision[:, 0])
    pd.DataFrame({"decision_margin": margin, "correct": best_predictions == y_test.to_numpy()}).to_csv("results/prediction_uncertainty.csv", index=False)
    plt.figure(figsize=(9, 5))
    plt.hist(margin, bins=20, edgecolor="white", color=STUDIO_COLORS["accent"])
    plt.xlabel("Decision margin")
    plt.ylabel("Test queries")
    plt.title("Prediction decision-margin distribution")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "prediction_decision_margin_distribution.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    Path("results/prediction_uncertainty.csv").write_text("metric,available\nconfidence_or_margin,false\n", encoding="utf-8")

experiment_summary = {
    "dataset_rows_total": int(len(train) + len(test)),
    "training_rows": int(len(train)),
    "test_rows": int(len(test)),
    "unique_intents_training": int(train[INTENT_COL].nunique()),
    "unique_intents_test": int(test[INTENT_COL].nunique()),
    "best_model": best_model_name,
    "test_accuracy": round(float(final_metrics["accuracy"]), 6),
    "test_macro_f1": round(float(final_metrics["macro_f1"]), 6),
    "majority_baseline_macro_f1": round(baseline_macro_f1, 6),
    "incorrect_test_predictions": int((best_predictions != y_test.to_numpy()).sum()),
    "leakage_status": "PASS" if len(final_overlap) == 0 and len(conflicts) == 0 else "REVIEW",
    "cv_folds": int(effective_folds),
    "random_seed": int(RANDOM_SEED),
}
Path("results/experiment_summary.json").write_text(json.dumps(experiment_summary, indent=2), encoding="utf-8")
print("Final analysis artifacts generated.")
print(json.dumps(experiment_summary, indent=2))


## 15. Final quality gate

The bundle is exported only after the current run passes the basic integrity checks below. This is a final guard against accidentally handing the dashboard incomplete or internally inconsistent results.


In [ ]:

# Export predictions before the final gate so the gate can verify the exact artifact it will package.
prediction_export = pd.DataFrame({
    "record_id": test["record_id"].astype(str).reset_index(drop=True),
    "true_intent": y_test.astype(str).reset_index(drop=True),
    "predicted_intent": pd.Series(best_predictions).astype(str).reset_index(drop=True),
})
prediction_export.to_csv("results/predictions.csv", index=False)

required_files = [
    Path("results/predictions.csv"), Path("results/final_metrics.json"),
    Path("results/model_comparison.csv"), Path("results/per_intent_performance.csv"),
    Path("results/error_analysis_sample.csv"), Path("results/audit_results.csv"),
    Path("results/leakage_audit.csv"), Path("results/experiment_summary.json"),
    Path("results/figures/intent_and_dataset_distribution.png"),
    Path("results/figures/query_length_histogram.png"),
    Path("results/figures/per_intent_f1.png"),
    Path("results/figures/confusion_matrix_best_model.png"),
    Path("results/figures/confusion_matrix_normalized.png"),
    Path("results/figures/model_comparison_cv_macro_f1.png"),
    Path("results/figures/baseline_vs_final_model.png"),
    Path("results/figures/data_quality_checks.png"),
]
missing = [str(p) for p in required_files if not p.exists() or p.stat().st_size == 0]
quality_checks = {
    "required_artifacts": len(missing) == 0,
    "predictions_match_test_rows": len(prediction_export) == len(test),
    "prediction_record_ids_unique": bool(prediction_export["record_id"].is_unique),
    "metric_accuracy_finite": bool(np.isfinite(float(final_metrics["accuracy"]))),
    "metric_macro_f1_finite": bool(np.isfinite(float(final_metrics["macro_f1"]))),
    "prediction_classes_valid": bool(set(map(str, best_predictions)).issubset(set(map(str, labels)))),
    "no_final_train_test_overlap": len(final_overlap) == 0,
    "no_conflicting_labels": len(conflicts) == 0,
    "portable_classifier_matches_sklearn": portable_parity_ok is not False,
}
print("FINAL QUALITY GATE")
print("-" * 60)
for name, passed in quality_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<6} {name}")
if missing:
    print("Missing artifacts:", missing)
if not all(quality_checks.values()):
    raise RuntimeError("Final quality gate failed. Fix the reported issue before packaging the dashboard bundle.")
Path("results/quality_gate.json").write_text(json.dumps({"status": "PASS", "checks": quality_checks}, indent=2), encoding="utf-8")
print("\nFINAL STATUS: PASS")


## 16. Export predictions and package the dashboard bundle

The final bundle now includes the evaluation artifacts, expanded figures, audit files, reproducibility metadata, and the quality-gate result. The Streamlit dashboard consumes this bundle without retraining the model. Raw CSV upload is intentionally kept inside Colab, so the dashboard receives only a completed experiment bundle.


In [ ]:
# predictions.csv was created and verified in the final quality-gate cell.


In [ ]:
# Model card: generated from this run's own variables. Nothing here is hand-written performance data.
_cmp = model_comparison.to_string(index=False)
_excluded = ", ".join(cv_excluded_classes) if cv_excluded_classes else "none"
_test_only = ", ".join(map(str, test_only_intents)) if test_only_intents else "none"
_sources = ", ".join(sorted({Path(d["path"]).name for d in DATASETS}))
model_card = f"""# Model card: {best_model_name}

Generated automatically by the Travel NLP Lab notebook. Values below come from this run only.

## Task
Single-label travel-query intent classification (one intent per query).

## Data
- Source file(s): {_sources}
- Training rows: {len(train):,} ({train[INTENT_COL].nunique()} intents)
- Held-out test rows: {len(test):,} ({test[INTENT_COL].nunique()} intents)
- Exact duplicate text/label rows were removed before splitting; train/test normalised-text overlap was checked.

## Model selection
- Candidates: TF-IDF + Logistic Regression, TF-IDF + Linear SVM, TF-IDF + Multinomial Naive Bayes.
- Method: stratified {effective_folds}-fold cross-validation on the TRAINING data only; metric = Macro-F1; seed = {RANDOM_SEED}.
- The test set was not used to choose the model. The selected model was fitted on the full training set and evaluated once on the test set.
- Classes excluded from cross-validation selection only (too few training rows): {_excluded}

```
{_cmp}
```

## Held-out test performance (single evaluation)
- Accuracy: {float(final_metrics['accuracy']):.4f}
- Macro-F1: {float(final_metrics['macro_f1']):.4f}
- Weighted-F1: {float(final_metrics['weighted_f1']):.4f}
- Intents present in test but never seen in training: {_test_only}

## Autocomplete (next-word) model
- Word n-gram model with stupid-backoff, built from training queries only; exported as results/next_word_model.json.
- Settings {next_word_model['selected_settings']} chosen on a validation slice of the training data; the test set was not used for tuning.
- Held-out next-word accuracy over {ev['positions']:,} word positions: top-1 {ev['top1_accuracy']:.1%}, top-3 {ev['top3_accuracy']:.1%}, top-5 {ev['top5_accuracy']:.1%} (word-frequency baseline top-3: {ev['frequent_words_baseline_top3']:.1%}).
- Fit check: {ev['fit_verdict']}.
- Suggestions reflect only the wording of this dataset; unseen words get no suggestion of their own.

## Limitations
- Macro-F1 is lower than accuracy when rare intents are poorly recognised; check per-intent results.
- Intents with very few examples give unstable per-intent metrics.
- Linear-model decision scores are not calibrated probabilities.
- Top terms per intent show feature association with predictions, not causal explanation.
- Results describe this dataset only and may not transfer to other travel domains, languages or phrasing.
- SHA-256 hashes in run_manifest.json show consistency with the manifest, not authorship or scientific validity.
"""
Path("model_card/model_card.md").write_text(model_card, encoding="utf-8")
print("Created model_card/model_card.md")

# Create a human-readable bundle README before packaging.
readme = f"""# Travel NLP Run Bundle\n\nThis bundle was generated by the Travel Query Intent Classification Using NLP notebook.\n\n## Run summary\n- Best model: {best_model_name}\n- Training rows: {len(train):,}\n- Test rows: {len(test):,}\n- Training intents: {train[INTENT_COL].nunique():,}\n- Test Accuracy: {float(final_metrics['accuracy']):.6f}\n- Test Macro-F1: {float(final_metrics['macro_f1']):.6f}\n- Cross-validation folds: {effective_folds}\n- Random seed: {RANDOM_SEED}\n\n## Main folders\n- `results/`: metrics, predictions, audits, errors, and figures\n- `data/raw/`: original input CSV files used for the run\n- `data/processed/`: processed train/test data\n- `model_card/`: model documentation generated by the notebook\n
## Dashboard handoff\n- Raw CSV files are uploaded and processed in Google Colab.\n- The Streamlit dashboard accepts only `travel_nlp_run_bundle.zip`; it does not train from or audit raw CSV uploads.\n\n## Important interpretation notes\n- The final test set is evaluated once, after training-only cross-validation selects the winning model.\n- Macro-F1 is the primary selection metric because it weights intents equally.\n- Linear-model decision scores are not probabilities.\n- SHA-256 values in `run_manifest.json` verify file contents associated with this run; they do not establish author identity or authenticity.\n"""
Path("results/BUNDLE_README.md").write_text(readme, encoding="utf-8")
print("Created results/BUNDLE_README.md")


In [ ]:
import glob
import hashlib
import json
import platform
import shutil
import uuid
from datetime import datetime, timezone

OUT = Path("travel_nlp_run_bundle")
shutil.rmtree(OUT, ignore_errors=True)

for source in ["results", "data/raw", "data/processed", "model_card"]:
    source_path = Path(source)
    if source_path.is_dir():
        shutil.copytree(source_path, OUT / source)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest_files = {}

for path in sorted(OUT.rglob("*")):
    if not path.is_file() or path.name == "run_manifest.json":
        continue

    relative = path.relative_to(OUT).as_posix()
    if relative.startswith(("results/", "data/raw/", "data/processed/", "model_card/")):
        manifest_files[relative] = {
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
        }

raw_hashes = [
    item["sha256"]
    for name, item in sorted(manifest_files.items())
    if name.startswith("data/raw/")
]
dataset_fingerprint = hashlib.sha256(
    "".join(raw_hashes).encode("utf-8")
).hexdigest()


import sys as _sys
def _ver(name):
    try:
        import importlib; return getattr(importlib.import_module(name), "__version__", "unknown")
    except Exception:
        return None
_has = lambda rel: any(n.startswith(rel) for n in manifest_files)
capabilities = {
    "predictions": _has("results/predictions.csv"),
    "confidence_margin": _has("results/prediction_uncertainty.csv"),
    "entities": any(n.endswith(".csv") and n.startswith("data/raw/") for n in manifest_files),
    "autocomplete": _has("results/next_word_model.json"),
    "live_classifier": _has("results/classifier_model.json"),
    "data_quality": _has("results/data_quality.json"),
    "figures": any(n.startswith("results/figures/") for n in manifest_files),
}
environment = {k: v for k, v in {
    "python": platform.python_version(), "platform": platform.platform(),
    "pandas": _ver("pandas"), "numpy": _ver("numpy"), "scikit-learn": _ver("sklearn"), "matplotlib": _ver("matplotlib"),
}.items() if v}
artifact_provenance = {
    "predictions": {"source": "notebook_benchmark", "note": "Produced once by this notebook on the held-out test set."},
    "classifier": {"source": "notebook_trained_model", "format": "travel_nlp_linear_v1" if capabilities["live_classifier"] else None},
    "entities": {"source": "dataset_annotation", "note": "Entity tags come from the dataset, not from a model."},
    "autocomplete": {"source": "dataset_derived", "note": "Word n-gram model built from the training queries."},
}

run_manifest = {
    "schema_version": 5,
    "pipeline_version": "8.2.0",
    "contract_version": "5.1",
    "run_id": str(uuid.uuid4()),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "python_version": platform.python_version(),
    "final_model": best_model_name,
    "model_selection": {
        "method": "stratified_cross_validation",
        "metric": "macro_f1",
        "folds": effective_folds,
    },
    "capabilities": capabilities,
    "environment": environment,
    "artifact_provenance": artifact_provenance,
    "dataset_fingerprint": dataset_fingerprint,
    "dataset_files": [
        name for name in sorted(manifest_files)
        if name.startswith("data/raw/")
    ],
    "files": manifest_files,
    "verification_note": (
        "SHA-256 hashes record the contents associated with this run manifest; "
        "they do not establish author identity or authenticity."
    ),
}

(OUT / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)

shutil.make_archive("travel_nlp_run_bundle", "zip", ".", OUT)

zip_path = Path("travel_nlp_run_bundle.zip")
if not zip_path.exists() or zip_path.stat().st_size == 0:
    raise RuntimeError("Bundle ZIP was not created correctly.")

print("Created travel_nlp_run_bundle.zip")
print("Final model:", best_model_name)
print("Dataset fingerprint:", dataset_fingerprint)

try:
    from google.colab import files
    files.download("travel_nlp_run_bundle.zip")
except ImportError:
    print("Colab download control is unavailable; the ZIP is saved in the current directory.")

## 17. Final handoff

The experiment is now ready for the Streamlit dashboard.

### Final workflow

`CSV dataset → Google Colab upload → NLP training/evaluation → travel_nlp_run_bundle.zip → Streamlit`

**Important:** raw CSV upload belongs in this Colab notebook. The Streamlit dashboard should accept **only** the completed `travel_nlp_run_bundle.zip`. It should not contain a raw CSV training/audit upload mode.

The bundle is the single source of experiment results for the dashboard. It contains the processed data, metrics, predictions, audits, errors, figures, provenance information, and integrity manifest generated by this run.

### Before submitting the project

Check:
- the dataset profile
- the class distribution
- the leakage audit
- the CV model-selection table
- the final test metrics
- the confusion matrix
- a few error examples
- the subgroup audit
- the live prediction section
- the generated `travel_nlp_run_bundle.zip`

This keeps the workflow reproducible and makes the separation clear: **Colab trains and evaluates; Streamlit verifies, visualizes, and presents the completed run.**


## 18. Submission checklist

Before sharing the project, make sure the notebook has been run from top to bottom and that `travel_nlp_run_bundle.zip` was created successfully. Raw CSV files belong in this Colab notebook; the Streamlit application receives the completed run bundle rather than raw CSV uploads.
